# convT-as-flipped-padded-conv — ex1: rebuild ConvTranspose2d as flipped padded Conv2d (stride 1)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `convT-as-flipped-padded-conv`. Running the final beacon cell reports progress against the `CNN: ConvT as flipped padded conv` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT as flipped padded conv` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convT-as-flipped-padded-conv`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convT-as-flipped-padded-conv"
DD_SUBTOPIC = "CNN: ConvT as flipped padded conv"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ConvTranspose as flipped-kernel padded-conv — quick refresher

`nn.ConvTranspose2d` looks unfamiliar but is just a regular convolution in disguise. For stride 1, no output_padding, kernel size `K`:

```
ConvTranspose2d(x, w)  ==  Conv2d(
    F.pad(x, (K-1, K-1, K-1, K-1)),     # add K-1 zero rows/cols on every side
    w.flip(-1).flip(-2).transpose(0, 1) # kernel-flip + axis-swap
)
```

**The three transforms:**
1. **Pad input by `K-1`** on every spatial side. This is why transpose-conv *expands* spatial dims — the output is `H_in + K - 1` (stride 1 case).
2. **Flip the kernel** along both spatial axes (`flip(-1)` then `flip(-2)`). This is the adjoint of cross-correlation.
3. **Swap channel axes** of the kernel (`transpose(0, 1)`) so its layout matches what a regular conv expects.

**Why this matters.** Many CNN papers describe "upsampling conv" or "deconvolution"; once you internalize the flipped-padded equivalence, all of the mystery dissolves and you can reason about its output shape with the same `(H + 2P - K) // S + 1` formula you already know.

(For stride > 1 there's an additional step of inserting `S - 1` zeros between each input pixel before padding — out of scope for this drill, we cover the stride-1 case.)

### Exercise 1 — rebuild ConvTranspose2d as flipped padded Conv2d (stride 1)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the equivalence between stride-1 `F.conv_transpose2d` and a regular `F.conv2d` on a `K-1`-padded input with a flipped-and-axis-swapped kernel.
> Keywords: ConvTranspose2d, flip-kernel, padding, adjoint
> ```

**KCs targeted:** `convT-padded-conv-equivalence`, `convT-kernel-flip-rule`

Implement `ex1_convT_as_padded_conv(x, weight)`. Given input `x: (B, IC, H, W)` and ConvTranspose2d weight `weight: (IC, OC, K, K)` (square kernel for simplicity), reproduce `F.conv_transpose2d(x, weight)` (stride 1, no padding, no output_padding) by:

1. **Padding `x` by `K - 1` on every spatial side** with zeros (use `F.pad(x, (K-1, K-1, K-1, K-1))`).
2. **Flipping the kernel** along both spatial axes (`weight.flip(-1).flip(-2)`).
3. **Swapping the kernel's channel axes** so it becomes `(OC, IC, K, K)` (`.transpose(0, 1)`).
4. Running `F.conv2d` on the padded input with the flipped+swapped kernel.

**Return** the resulting tensor; it must equal `F.conv_transpose2d(x, weight)` to fp tolerance.

**The point of the drill.** ConvTranspose is just regular conv in disguise. Once you internalize the three transforms (pad, flip, swap), all of its mysteries dissolve.

Hint on shape: the output is `(B, OC, H + K - 1, W + K - 1)` (stride 1, no extra padding). Transpose-conv *expands* spatial dims — that's why it's the canonical upsampling op.

In [ ]:
def ex1_convT_as_padded_conv(x: Tensor, weight: Tensor) -> Tensor:
    from torch.nn import functional as F
    K = weight.shape[-1]
    x_pad = F.pad(x, (K - 1, K - 1, K - 1, K - 1))           # (B, IC, H+2(K-1), W+2(K-1))
    w_flipped = weight.flip(-1).flip(-2)                     # spatial flip
    w_swapped = w_flipped.transpose(0, 1)                    # (OC, IC, K, K)
    return F.conv2d(x_pad, w_swapped)


<details><summary>Solution</summary>

```python
def ex1_convT_as_padded_conv(x: Tensor, weight: Tensor) -> Tensor:
    from torch.nn import functional as F
    K = weight.shape[-1]
    x_pad = F.pad(x, (K - 1, K - 1, K - 1, K - 1))           # (B, IC, H+2(K-1), W+2(K-1))
    w_flipped = weight.flip(-1).flip(-2)                     # spatial flip
    w_swapped = w_flipped.transpose(0, 1)                    # (OC, IC, K, K)
    return F.conv2d(x_pad, w_swapped)
```

**Why pad by `K - 1`.** A regular `(K, K)` conv on input of size `(H, W)` produces output `(H - K + 1, W - K + 1)`. To get transpose-conv's *expanded* output `(H + K - 1, W + K - 1)`, we need an effective input of `(H + 2(K-1), W + 2(K-1))` — i.e., `K-1` padding on every side.

**Why flip both spatial axes.** The mathematical adjoint of cross-correlation (what PyTorch calls 'conv') is correlation with the flipped kernel. The flip swaps the role of the kernel indices `(kh, kw) ↔ (K-1-kh, K-1-kw)`, which is exactly what the adjoint operation requires when you derive it from the summation form.

**Why swap channel axes.** ConvTranspose2d stores `(IC, OC, K, K)`, but `F.conv2d` expects `(OC, IC, K, K)`. The `.transpose(0, 1)` is purely a layout reformat — no flipping involved, just an axis relabel. Forgetting this raises a `size mismatch`.

**Stride > 1 generalization.** For `stride = S` you ALSO need to insert `S - 1` zero rows/columns between every input pixel *before* padding by `K - 1`. That's where the upsampling factor comes from. We restrict to stride 1 here to keep the drill on the three-transforms story; the stride extension is a strictly harder follow-up.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()